In [1]:
import scanpy as sc

/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel fro

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
from scipy import sparse
import gc
import os
warnings.filterwarnings('ignore')

def optimize_adata_memory(adata, name=""):
    """深度优化AnnData内存使用"""
    print(f"优化 {name}: {adata.n_obs} cells, {adata.n_vars} genes")
    
    # 1. 确保X是稀疏矩阵
    if not sparse.issparse(adata.X):
        adata.X = sparse.csr_matrix(adata.X)
        print("  - X转换为稀疏矩阵")
    
    # 2. 删除不必要的部分
    if hasattr(adata, 'raw') and adata.raw is not None:
        del adata.raw
        print("  - 删除raw数据")
    
    # 3. 清理不必要的layers
    unnecessary_layers = ['spliced', 'unspliced']
    for layer in list(adata.layers.keys()):
        if layer in unnecessary_layers:
            del adata.layers[layer]
            print(f"  - 删除layer: {layer}")
    
    # 4. 优化obs和var数据类型
    for col in adata.obs.columns:
        if adata.obs[col].dtype == 'object':
            unique_ratio = adata.obs[col].nunique() / len(adata.obs)
            if unique_ratio < 0.5:  # 如果唯一值比例小于50%，转换为category
                adata.obs[col] = adata.obs[col].astype('category')
    
    for col in adata.var.columns:
        if adata.var[col].dtype == 'object':
            adata.var[col] = adata.var[col].astype('category')
    
    # 5. 强制垃圾回收
    gc.collect()
    
    return adata

def load_previously_sampled_cells(previous_sampled_path):
    """加载先前抽取的50万细胞信息"""
    print("加载先前抽取的细胞信息...")
    prev_sampled = sc.read_h5ad(previous_sampled_path, backed='r')
    prev_cell_ids = set(prev_sampled.obs_names)
    print(f"先前抽取的细胞数: {len(prev_cell_ids)}")
    
    # 检查数据集组成
    if 'dataset' in prev_sampled.obs.columns:
        dataset_composition = prev_sampled.obs['dataset'].value_counts()
        print("先前抽取的数据集组成:")
        for dataset, count in dataset_composition.items():
            print(f"  - {dataset}: {count} cells")
    
    del prev_sampled
    gc.collect()
    
    return prev_cell_ids

def simple_sampling_from_remaining(adata, target_cells, prev_cell_ids, dataset_name):
    """从剩余细胞中简单随机抽样"""
    
    # 获取当前数据集的所有细胞ID
    all_cell_ids = set(adata.obs_names)
    
    # 排除先前已抽取的细胞
    remaining_cell_ids = all_cell_ids - prev_cell_ids
    print(f"{dataset_name}: 总细胞{len(all_cell_ids)}, 已抽取{len(prev_cell_ids & all_cell_ids)}, 剩余{len(remaining_cell_ids)}")
    
    if len(remaining_cell_ids) == 0:
        print(f"警告: {dataset_name}没有剩余细胞可抽取")
        return None
    
    # 如果剩余细胞少于目标数，全取
    if len(remaining_cell_ids) <= target_cells:
        sampled_cell_ids = remaining_cell_ids
        print(f"  - {dataset_name}: 剩余细胞不足，全取{len(sampled_cell_ids)}个细胞")
    else:
        # 随机抽样
        sampled_cell_ids = set(np.random.choice(list(remaining_cell_ids), target_cells, replace=False))
        print(f"  - {dataset_name}: 从剩余细胞中随机抽取{len(sampled_cell_ids)}个细胞")
    
    # 提取抽样细胞
    adata_sampled = adata[list(sampled_cell_ids)].copy()
    
    return adata_sampled

def prepare_orig_ident_column(adata, dataset_name):
    """准备orig.ident列，确保所有数据集都有统一的orig.ident列"""
    print(f"准备 {dataset_name} 的orig.ident列")
    
    if 'orig.ident' in adata.obs.columns:
        print(f"  - 已存在orig.ident列")
        adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)
    else:
        if 'orig.ident' in adata.obs.columns:
            adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)
            print(f"  - {dataset_name}数据: orig.ident.ident")
        else:
            adata.obs['orig.ident'] = dataset_name
            print(f"  - {dataset_name}数据: 使用数据集名称作为orig.ident")
    
    print(f"  - orig.ident列数据类型: {adata.obs['orig.ident'].dtype}")
    print(f"  - orig.ident唯一值: {adata.obs['orig.ident'].nunique()}")
    
    return adata

# 设置随机种子
np.random.seed(42)

# 文件路径
adult_path = "/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/data/human_adult_brain_clean.h5ad"
fetal_path = "/storage2/liuxiaodongLab/fanxueying/developmental_atlas/code/20251107_human_fetal_brain_data_prep/human_fetal_brain_clean.h5ad"
previous_sampled_path = "/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/filter_defined_number_combined_sampled.h5ad"

# 1. 加载先前抽取的50万细胞信息
prev_cell_ids = load_previously_sampled_cells(previous_sampled_path)

# 2. 计算共同基因
print("计算共同基因...")
fetal_genes = set(sc.read_h5ad(fetal_path, backed='r').var_names)
adult_genes = set(sc.read_h5ad(adult_path, backed='r').var_names)
common_genes = fetal_genes & adult_genes
print(f"共同基因: {len(common_genes)}")

# 3. 目标细胞数
target_total = 100000  # 10万细胞

# 先获取adult和fetal的剩余细胞数
print("\n评估剩余细胞情况...")
adult_adata = sc.read_h5ad(adult_path, backed='r')
fetal_adata = sc.read_h5ad(fetal_path, backed='r')

adult_remaining = len(set(adult_adata.obs_names) - prev_cell_ids)
fetal_remaining = len(set(fetal_adata.obs_names) - prev_cell_ids)
total_remaining = adult_remaining + fetal_remaining

print(f"Adult剩余细胞: {adult_remaining}")
print(f"Fetal剩余细胞: {fetal_remaining}")
print(f"总剩余细胞: {total_remaining}")

# 按比例分配目标细胞数
if total_remaining > 0:
    adult_target = int(target_total * (adult_remaining / total_remaining))
    fetal_target = target_total - adult_target
else:
    adult_target = target_total // 2
    fetal_target = target_total - adult_target

print(f"\n目标分配:")
print(f"  - 总目标: {target_total}细胞")
print(f"  - Adult目标: {adult_target}细胞")
print(f"  - Fetal目标: {fetal_target}细胞")

del adult_adata, fetal_adata
gc.collect()

# 4. 处理每个数据集
print("\n开始抽样...")
datasets = []

# 处理Adult数据
print("\n处理Adult数据...")
adult_adata = sc.read_h5ad(adult_path)
adult_adata = prepare_orig_ident_column(adult_adata, 'adult')

# lineage处理
if 'lineage_pred' in adult_adata.obs.columns:
    adult_adata.obs['lineage'] = adult_adata.obs['lineage_pred'].copy()

adult_sampled = simple_sampling_from_remaining(adult_adata, adult_target, prev_cell_ids, 'Adult')

if adult_sampled is not None:
    # 只保留共同基因
    genes_to_keep = list(set(adult_sampled.var_names) & common_genes)
    adult_sampled = adult_sampled[:, genes_to_keep].copy()
    adult_sampled = optimize_adata_memory(adult_sampled, "Adult抽样后")
    adult_sampled.obs['dataset'] = 'adult'
    datasets.append(adult_sampled)

del adult_adata, adult_sampled
gc.collect()

# 处理Fetal数据
print("\n处理Fetal数据...")
fetal_adata = sc.read_h5ad(fetal_path)
fetal_adata = prepare_orig_ident_column(fetal_adata, 'fetal')

# lineage处理
if 'lineage_pred' in fetal_adata.obs.columns:
    fetal_adata.obs['lineage'] = fetal_adata.obs['lineage_pred'].copy()

fetal_sampled = simple_sampling_from_remaining(fetal_adata, fetal_target, prev_cell_ids, 'Fetal')

if fetal_sampled is not None:
    # 只保留共同基因
    genes_to_keep = list(set(fetal_sampled.var_names) & common_genes)
    fetal_sampled = fetal_sampled[:, genes_to_keep].copy()
    fetal_sampled = optimize_adata_memory(fetal_sampled, "Fetal抽样后")
    fetal_sampled.obs['dataset'] = 'fetal'
    datasets.append(fetal_sampled)

del fetal_adata, fetal_sampled
gc.collect()

# 5. 合并数据集
if datasets:
    print("\n合并数据集...")
    combined = sc.concat(datasets, join='inner')
    combined = optimize_adata_memory(combined, "最终合并")
    
    print(f"最终数据集: {combined.n_obs} cells, {combined.n_vars} genes")
    
    # 修复数据格式问题
    print("修复数据格式问题...")
    for col in combined.obs.columns:
        if pd.api.types.is_categorical_dtype(combined.obs[col]):
            combined.obs[col] = combined.obs[col].astype(str)
    
    # 检查并修复数值列
    numeric_columns = ['percent.mt', 'n_genes', 'n_counts', 'total_counts']
    for col in numeric_columns:
        if col in combined.obs.columns:
            combined.obs[col] = pd.to_numeric(combined.obs[col], errors='coerce')
    
    # 确保orig.ident列存在且是字符串类型
    if 'orig.ident' in combined.obs.columns:
        combined.obs['orig.ident'] = combined.obs['orig.ident'].astype(str)
        print("orig.ident列信息:")
        print(combined.obs['orig.ident'].value_counts())
    
    # 6. 保存两个版本
    print("\n保存结果...")
    
    # 版本1: 包含lineage列
    output_with_lineage = "/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/validation_100k_with_lineage.h5ad"
    combined.write_h5ad(output_with_lineage)
    print(f"保存包含lineage的数据到: {output_with_lineage}")
    
    # 版本2: 不包含lineage列
    combined_no_lineage = combined.copy()
    if 'lineage' in combined_no_lineage.obs.columns:
        combined_no_lineage.obs = combined_no_lineage.obs.drop(columns=['lineage'])
        print("已移除lineage列")
    
    output_without_lineage = "/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/validation_100k_without_lineage.h5ad"
    combined_no_lineage.write_h5ad(output_without_lineage)
    print(f"保存不包含lineage的数据到: {output_without_lineage}")
    
    # 输出数据集统计信息
    print(f"\n最终统计:")
    print(f"总细胞数: {combined.n_obs}")
    if 'dataset' in combined.obs.columns:
        dataset_stats = combined.obs['dataset'].value_counts()
        print("数据集分布:")
        for dataset, count in dataset_stats.items():
            print(f"  - {dataset}: {count} cells")
    
    if 'lineage' in combined.obs.columns:
        lineage_stats = combined.obs['lineage'].value_counts()
        print("Lineage分布 (前10):")
        for lineage, count in lineage_stats.head(10).items():
            print(f"  - {lineage}: {count} cells")
    
else:
    print("错误: 没有成功抽取到任何细胞")

print("处理完成!")

/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel fro

加载先前抽取的细胞信息...
先前抽取的细胞数: 437040
先前抽取的数据集组成:
  - adult: 232115 cells
  - fetal: 169748 cells
  - pre: 35177 cells
计算共同基因...
共同基因: 37540

评估剩余细胞情况...
Adult剩余细胞: 3137104
Fetal剩余细胞: 1496189
总剩余细胞: 4633293

目标分配:
  - 总目标: 100000细胞
  - Adult目标: 67707细胞
  - Fetal目标: 32293细胞

开始抽样...

处理Adult数据...
准备 adult 的orig.ident列
  - 已存在orig.ident列
  - orig.ident列数据类型: object
  - orig.ident唯一值: 1
Adult: 总细胞3369219, 已抽取232115, 剩余3137104
  - Adult: 从剩余细胞中随机抽取67707个细胞
优化 Adult抽样后: 67707 cells, 37540 genes

处理Fetal数据...
准备 fetal 的orig.ident列
  - 已存在orig.ident列
  - orig.ident列数据类型: object
  - orig.ident唯一值: 1
Fetal: 总细胞1665937, 已抽取169748, 剩余1496189
  - Fetal: 从剩余细胞中随机抽取32293个细胞
优化 Fetal抽样后: 32293 cells, 37540 genes

合并数据集...
优化 最终合并: 100000 cells, 37540 genes
最终数据集: 100000 cells, 37540 genes
修复数据格式问题...
orig.ident列信息:
orig.ident
Kimberly Siletti    67707
Braun               32293
Name: count, dtype: int64

保存结果...
保存包含lineage的数据到: /storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_f